# LeetCode 2899 — Last Visited Integers bằng Python

Link đề: https://leetcode.com/problems/last-visited-integers/ · Độ khó: **Easy** · Chủ đề: Array, Simulation

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def lastVisitedIntegers(self, nums: List[int]) -> List[int]:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: duyệt mảng `nums` gồm số nguyên dương và `-1`; mỗi khi gặp `-1`, phải tìm "số nguyên dương được thăm gần nhất" theo một quy tắc đếm các `-1` liên tiếp.
2. Hiểu và dùng được **danh sách `seen` "mới nhất đứng đầu"**, **chỉ số âm** (`seen[-k]`), và biến đếm `k` các `-1` **liên tiếp** (đặt lại về 0 khi gặp số dương).
3. Đi từ cách **mô phỏng đúng như đề (chèn vào đầu danh sách)** → **thêm vào cuối và dùng chỉ số âm** → **con trỏ lùi, không cần đếm `k`** → **`itertools.groupby` xử lý từng đoạn `-1`**.
4. Biết vì sao `insert(0, x)` chậm hơn `append(x)` và cách đổi chiều danh sách để tránh chèn đầu.
5. Tránh các lỗi kinh điển: quên đặt lại `k` khi gặp số dương, nhầm `seen[k]` với `seen[k - 1]`, quên điều kiện `k > len(seen)` (ra `-1`), và đếm `-1` tính cả những `-1` không liên tiếp.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`, mỗi phần tử hoặc là **số nguyên dương** hoặc là **`-1`**. Ta cần tìm, cho **mỗi** `-1`, "số nguyên dương được thăm gần nhất" của nó.

Hãy tạo hai mảng rỗng `seen` và `ans` rồi duyệt `nums` từ đầu đến cuối:

- Gặp **số nguyên dương** `x`: **chèn `x` vào đầu** `seen` (số mới nhất đứng đầu).
- Gặp **`-1`**: gọi `k` là **số lượng `-1` liên tiếp** đã gặp tính đến thời điểm này (**gồm cả `-1` hiện tại**).
  - Nếu `k <= len(seen)`: thêm **phần tử thứ `k`** của `seen` (đánh số từ 1, tức `seen[k - 1]`) vào `ans`.
  - Ngược lại (`k > len(seen)`): thêm `-1` vào `ans`.

Trả về mảng `ans`.

**Ví dụ 1**

- Input: `nums = [1, 2, -1, -1, -1]`
- Output: `[2, 1, -1]`
- Giải thích: sau `1, 2` thì `seen = [2, 1]`. `-1` đầu tiên (`k = 1`) lấy phần tử thứ 1 là `2`. `-1` thứ hai (`k = 2`) lấy phần tử thứ 2 là `1`. `-1` thứ ba (`k = 3 > len(seen) = 2`) cho `-1`.

**Ví dụ 2**

- Input: `nums = [1, -1, 2, -1, -1]`
- Output: `[1, 2, 1]`
- Giải thích: `seen = [1]`; `-1` (`k = 1`) lấy `1`. Gặp `2` thì `seen = [2, 1]` và dãy `-1` liên tiếp bị cắt (đếm lại từ 0). `-1` (`k = 1`) lấy `2`; `-1` tiếp (`k = 2`) lấy `1`.

**Ràng buộc:** `1 <= nums.length <= 100`; `nums[i] == -1` hoặc `1 <= nums[i] <= 100`.

In [1]:
from typing import List
from collections import deque
from itertools import groupby
import random

examples = [
    ([1, 2, -1, -1, -1], [2, 1, -1]),
    ([1, -1, 2, -1, -1], [1, 2, 1]),
]
for nums, expected in examples:
    print("nums =", nums, "-> kỳ vọng", expected)

nums = [1, 2, -1, -1, -1] -> kỳ vọng [2, 1, -1]
nums = [1, -1, 2, -1, -1] -> kỳ vọng [1, 2, 1]


## 3. Kiến thức nền

### 3.1. Danh sách `list`: `append`, `insert`, chỉ số

- `a.append(x)`: thêm `x` vào **cuối**; rất nhanh.
- `a.insert(0, x)`: thêm `x` vào **đầu**; phải **dịch chuyển mọi phần tử** sang phải nên chậm với danh sách dài (`O(len(a))`).
- `a[i]`: phần tử ở vị trí `i` (đếm từ **0**).
- `len(a)`: số phần tử.

Đề nói "phần tử thứ `k`" theo cách đếm từ **1** (người thường đếm), còn Python đếm từ **0**, nên phần tử thứ `k` là `seen[k - 1]`.

In [2]:
seen = []
seen.insert(0, 1)       # [1]
seen.insert(0, 2)       # [2, 1]: số mới nhất đứng đầu
print(seen)
print(seen[0], seen[1])  # phần tử thứ 1 và thứ 2 (đếm từ 1) là seen[0], seen[1]

b = []
b.append(1)
b.append(2)
print(b)                # [1, 2]: số mới nhất đứng cuối

[2, 1]
2 1
[1, 2]


### 3.2. Chỉ số âm: `a[-1]`, `a[-2]`

Python cho phép chỉ số **âm** để đếm từ **cuối**: `a[-1]` là phần tử cuối, `a[-2]` là phần tử áp cuối... Hợp với danh sách mà số mới nhất nằm **ở cuối** (dùng `append`): phần tử mới nhất thứ `k` là `a[-k]`. Chỉ số âm hợp lệ khi `k <= len(a)`; nếu vượt, Python báo `IndexError`.

In [3]:
a = [10, 20, 30, 40]
print(a[-1], a[-2], a[-4])       # 40 30 10
k = 2
print("phần tử mới nhất thứ", k, "=", a[-k])    # 30
try:
    print(a[-5])
except IndexError as e:
    print("IndexError:", e)

40 30 10
phần tử mới nhất thứ 2 = 30
IndexError: list index out of range


### 3.3. Biến đếm các số `-1` liên tiếp

Ta cần **đếm số `-1` liên tiếp**: mỗi `-1` làm `k` tăng 1; **gặp một số dương thì `k` đặt lại về 0** (chuỗi liên tiếp bị cắt). Ví dụ với `[1, -1, 2, -1, -1]`: `k` lần lượt là `0, 1, 0, 1, 2`.

In [4]:
nums = [1, -1, 2, -1, -1]
k = 0
for x in nums:
    if x == -1:
        k += 1
    else:
        k = 0
    print(f"x={x:>2} -> k={k}")

x= 1 -> k=0
x=-1 -> k=1
x= 2 -> k=0
x=-1 -> k=1
x=-1 -> k=2


### 3.4. Sao chép/đảo danh sách: `reversed` và cắt lát `[::-1]`

`a[::-1]` cho danh sách **đảo ngược** (bản mới). Nếu ta lưu số theo thứ tự xuất hiện (`append`), thì `a[::-1]` có đúng dạng "mới nhất đứng đầu" mà đề gọi là `seen`.

In [5]:
order = [1, 2, 3]                 # thứ tự xuất hiện
print(order[::-1])                # [3, 2, 1] = seen theo đề
print(list(reversed(order)))      # cách khác
print(order[::-1][0] == order[-1])  # phần tử đầu của seen chính là phần tử cuối của order

[3, 2, 1]
[3, 2, 1]
True


### 3.5. `deque` và `appendleft`

`collections.deque` là hàng đợi hai đầu: `appendleft(x)` thêm vào **đầu** trong `O(1)`, nên dùng được thay cho `list.insert(0, x)`. Nhược điểm: truy cập phần tử ở **giữa** chậm (`O(n)`), nhưng ở bài này `k` thường nhỏ.

In [6]:
d = deque()
d.appendleft(1)
d.appendleft(2)
print(list(d), d[0], d[1])       # [2, 1] 2 1

[2, 1] 2 1


### 3.6. `itertools.groupby`: gom các phần tử liên tiếp giống nhau theo khóa

`groupby(iterable, key=hàm)` chia dãy thành các **đoạn liên tiếp** có cùng giá trị khóa. Với `key=lambda x: x == -1`, các đoạn là "toàn `-1`" hoặc "toàn số dương" xen kẽ. Dùng để xử lý **từng đoạn `-1`** một lần.

In [7]:
nums = [1, 2, -1, -1, 3, -1]
for is_minus, run in groupby(nums, key=lambda x: x == -1):
    print("toàn -1" if is_minus else "toàn dương", list(run))

toàn dương [1, 2]
toàn -1 [-1, -1]
toàn dương [3]
toàn -1 [-1]


### 3.7. Biểu thức điều kiện (`a if cond else b`) để chọn giá trị hoặc `-1`

Khi `k <= len(seen)` ta lấy `seen[k - 1]`, ngược lại lấy `-1`. Viết gọn thành một biểu thức: `seen[k - 1] if k <= len(seen) else -1`. Nhớ kiểm tra điều kiện **trước** khi truy cập chỉ số để không bị `IndexError`.

In [8]:
seen = [2, 1]
for k in [1, 2, 3]:
    value = seen[k - 1] if k <= len(seen) else -1
    print("k =", k, "->", value)

k = 1 -> 2
k = 2 -> 1
k = 3 -> -1


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hình dung bạn đang **lật từng trang sách**: trang dương là "trang đã đọc", `-1` là lệnh "quay lại": lần quay đầu về trang đọc **gần nhất**, lệnh `-1` liên tiếp thứ hai về trang **trước đó nữa**, và cứ thế. Khi đọc thêm một trang mới thì "bộ đếm quay lại" được đặt lại.

Ví dụ 2: `nums = [1, -1, 2, -1, -1]`

| bước | x | `seen` (mới nhất đứng đầu) | `k` | kết quả thêm vào `ans` |
|---|---|---|---|---|
| 1 | 1 | `[1]` | 0 | (không) |
| 2 | -1 | `[1]` | 1 | `seen[0] = 1` |
| 3 | 2 | `[2, 1]` | 0 | (không) |
| 4 | -1 | `[2, 1]` | 1 | `seen[0] = 2` |
| 5 | -1 | `[2, 1]` | 2 | `seen[1] = 1` |

`ans = [1, 2, 1]`. Quy luật: **`k` là "độ lùi" từ số mới nhất**; vượt quá số lượng số đã thấy thì không còn gì để lùi, trả `-1`.

## 5. Cách 1 — Brute Force / baseline: mô phỏng đúng như đề (`insert(0, x)`)

**Ý tưởng.** Làm đúng từng chữ của đề: `seen.insert(0, x)` cho số dương; với `-1` thì tăng `k` rồi tra `seen[k - 1]` hoặc `-1`; số dương thì `k = 0`.

In [9]:
def last_visited_literal(nums: List[int]) -> List[int]:
    seen = []                                     # mới nhất đứng đầu
    ans = []
    k = 0                                         # số -1 liên tiếp (gồm -1 hiện tại)
    for x in nums:
        if x == -1:
            k += 1
            ans.append(seen[k - 1] if k <= len(seen) else -1)
        else:
            seen.insert(0, x)                     # chèn vào ĐẦU
            k = 0                                 # chuỗi -1 bị cắt
    return ans


for nums, expected in examples:
    print(nums, "->", last_visited_literal(nums), "(kỳ vọng", expected, ")")

[1, 2, -1, -1, -1] -> [2, 1, -1] (kỳ vọng [2, 1, -1] )
[1, -1, 2, -1, -1] -> [1, 2, 1] (kỳ vọng [1, 2, 1] )


In [10]:
# Quan sát seen và k thay đổi theo từng bước với ví dụ 1
nums = [1, 2, -1, -1, -1]
seen, k = [], 0
for x in nums:
    if x == -1:
        k += 1
        print(f"x=-1: k={k}, seen={seen} -> {'seen[%d]=%s' % (k - 1, seen[k - 1]) if k <= len(seen) else 'k > len(seen) -> -1'}")
    else:
        seen.insert(0, x)
        k = 0
        print(f"x={x}: seen={seen}, k=0")

x=1: seen=[1], k=0
x=2: seen=[2, 1], k=0
x=-1: k=1, seen=[2, 1] -> seen[0]=2
x=-1: k=2, seen=[2, 1] -> seen[1]=1
x=-1: k=3, seen=[2, 1] -> k > len(seen) -> -1


**Độ phức tạp (Cách 1).** Gọi `n` là độ dài `nums`. Mỗi `insert(0, x)` tốn `O(len(seen))`, tổng tối đa `O(n²)`; mỗi `-1` tra `O(1)`: **Time `O(n²)`** trong trường hợp xấu nhất; `seen` và `ans` tối đa `n` phần tử: **Space `O(n)`**.

**Nút thắt.** `insert(0, x)` dịch chuyển cả danh sách. Với `n <= 100` không đáng kể, nhưng ta có thể bỏ hẳn việc chèn đầu bằng cách đổi chiều danh sách.

## 6. Ý tưởng thuật toán / bước đột phá

Hai quan sát:

1. **Chiều danh sách chỉ là quy ước.** Nếu ta lưu số theo thứ tự xuất hiện bằng `append` (số mới nhất ở **cuối**), thì "phần tử thứ `k` của `seen`" (đếm từ đầu, mới nhất đứng đầu) chính là **phần tử thứ `k` tính từ cuối**: `order[-k]`. Không cần `insert(0, ...)`.
2. **Không cần đếm `k`.** Mỗi lần gặp `-1` trong cùng một đoạn, ta chỉ cần **lùi con trỏ thêm một ô**: đặt `ptr = len(order) - 1` khi gặp số dương, và mỗi `-1` thì `ans.append(order[ptr] if ptr >= 0 else -1)` rồi `ptr -= 1`.

Cả hai cho kết quả giống hệt mô phỏng gốc nhưng mỗi phần tử xử lý trong `O(1)`.

## 7. Các cách tiếp theo

### Cách 2 — `append` vào cuối + chỉ số âm `order[-k]`

**Giải thích.** `order` lưu theo thứ tự xuất hiện. Khi gặp `-1` thứ `k` trong đoạn: nếu `k <= len(order)` thì thêm `order[-k]` (phần tử mới nhất thứ `k`), ngược lại thêm `-1`. Vẫn cần đếm `k`.

In [11]:
def last_visited_append(nums: List[int]) -> List[int]:
    order = []                                    # thứ tự xuất hiện, mới nhất ở CUỐI
    ans = []
    k = 0
    for x in nums:
        if x == -1:
            k += 1
            ans.append(order[-k] if k <= len(order) else -1)
        else:
            order.append(x)                       # append: O(1)
            k = 0
    return ans


for nums, expected in examples:
    print(nums, "->", last_visited_append(nums), "(kỳ vọng", expected, ")")

[1, 2, -1, -1, -1] -> [2, 1, -1] (kỳ vọng [2, 1, -1] )
[1, -1, 2, -1, -1] -> [1, 2, 1] (kỳ vọng [1, 2, 1] )


In [12]:
# Code nộp LeetCode cho Cách 2
from typing import List

class Solution:
    def lastVisitedIntegers(self, nums: List[int]) -> List[int]:
        order = []
        ans = []
        k = 0
        for x in nums:
            if x == -1:
                k += 1
                ans.append(order[-k] if k <= len(order) else -1)
            else:
                order.append(x)
                k = 0
        return ans

**Độ phức tạp Cách 2:** **Time `O(n)`** (mỗi phần tử `O(1)`), **Space `O(n)`**.

### Cách 3 — Con trỏ lùi `ptr` (không cần đếm `k`)

**Giải thích.** Mỗi khi gặp số dương, thêm vào `order` và đặt `ptr = len(order) - 1` (trỏ vào số mới nhất). Mỗi `-1`: nếu `ptr >= 0` thì trả `order[ptr]`, ngược lại `-1`; rồi `ptr -= 1`. Khi `ptr` xuống dưới `0` mọi `-1` tiếp theo đều cho `-1` (không còn gì để lùi).

In [13]:
def last_visited_pointer(nums: List[int]) -> List[int]:
    order = []
    ans = []
    ptr = -1                                      # chưa có số nào
    for x in nums:
        if x == -1:
            ans.append(order[ptr] if ptr >= 0 else -1)
            ptr -= 1                              # lùi một ô cho -1 tiếp theo
        else:
            order.append(x)
            ptr = len(order) - 1                  # trỏ vào số mới nhất, bắt đầu đoạn mới
    return ans


for nums, expected in examples:
    print(nums, "->", last_visited_pointer(nums), "(kỳ vọng", expected, ")")

[1, 2, -1, -1, -1] -> [2, 1, -1] (kỳ vọng [2, 1, -1] )
[1, -1, 2, -1, -1] -> [1, 2, 1] (kỳ vọng [1, 2, 1] )


In [14]:
# Code nộp LeetCode cho Cách 3
from typing import List

class Solution:
    def lastVisitedIntegers(self, nums: List[int]) -> List[int]:
        order = []
        ans = []
        ptr = -1
        for x in nums:
            if x == -1:
                ans.append(order[ptr] if ptr >= 0 else -1)
                ptr -= 1
            else:
                order.append(x)
                ptr = len(order) - 1
        return ans

**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(n)`**.

### Cách 4 — `itertools.groupby`: xử lý từng đoạn `-1`

**Giải thích.** Gom `nums` thành các đoạn "toàn số dương" và "toàn `-1`". Với đoạn dương, thêm vào `order`. Với đoạn `-1` dài `L`: các đáp án là `L` số mới nhất theo thứ tự lùi, tức `order[::-1][:L]`, **đệm thêm `-1`** nếu `L` lớn hơn số đã thấy. Cách này nhìn bài như "lấy `L` phần tử đầu của `seen`".

In [15]:
def last_visited_groupby(nums: List[int]) -> List[int]:
    order = []
    ans = []
    for is_minus, run in groupby(nums, key=lambda x: x == -1):
        run = list(run)
        if is_minus:
            recent = order[::-1][:len(run)]                  # seen[:L]
            ans.extend(recent + [-1] * (len(run) - len(recent)))
        else:
            order.extend(run)
    return ans


for nums, expected in examples:
    print(nums, "->", last_visited_groupby(nums), "(kỳ vọng", expected, ")")

[1, 2, -1, -1, -1] -> [2, 1, -1] (kỳ vọng [2, 1, -1] )
[1, -1, 2, -1, -1] -> [1, 2, 1] (kỳ vọng [1, 2, 1] )


**Độ phức tạp Cách 4:** **Time `O(n)`** nếu cắt `order[::-1][:L]` chỉ sao chép phần tử cần thiết (mỗi lần tạo bản đảo `O(len(order))`, tổng tệ nhất `O(n²)`; với `n <= 100` không đáng kể), **Space `O(n)`**.

> Cách 3 vừa ngắn vừa tối ưu; Cách 2 dễ hiểu nhất sau Cách 1. Cả hai đều có thể nộp.

## 8. TRACE từng bước

Ta theo dõi Cách 3 (con trỏ) và Cách 2 (chỉ số âm) qua ví dụ `[1, 2, -1, -1, -1, 3, -1, -1]` có cả hai đoạn `-1`.

In [16]:
def trace_pointer(nums):
    order, ans, ptr = [], [], -1
    print("nums =", nums)
    for i, x in enumerate(nums):
        if x == -1:
            value = order[ptr] if ptr >= 0 else -1
            print(f"i={i}: x=-1 | order={order} ptr={ptr} -> thêm {value} vào ans; ptr -> {ptr - 1}")
            ans.append(value)
            ptr -= 1
        else:
            order.append(x)
            ptr = len(order) - 1
            print(f"i={i}: x={x:>2} | thêm vào order -> {order}; ptr đặt = {ptr}")
    print("ans =", ans)
    return ans


assert trace_pointer([1, 2, -1, -1, -1, 3, -1, -1]) == [2, 1, -1, 3, 2]

nums = [1, 2, -1, -1, -1, 3, -1, -1]
i=0: x= 1 | thêm vào order -> [1]; ptr đặt = 0
i=1: x= 2 | thêm vào order -> [1, 2]; ptr đặt = 1
i=2: x=-1 | order=[1, 2] ptr=1 -> thêm 2 vào ans; ptr -> 0
i=3: x=-1 | order=[1, 2] ptr=0 -> thêm 1 vào ans; ptr -> -1
i=4: x=-1 | order=[1, 2] ptr=-1 -> thêm -1 vào ans; ptr -> -2
i=5: x= 3 | thêm vào order -> [1, 2, 3]; ptr đặt = 2
i=6: x=-1 | order=[1, 2, 3] ptr=2 -> thêm 3 vào ans; ptr -> 1
i=7: x=-1 | order=[1, 2, 3] ptr=1 -> thêm 2 vào ans; ptr -> 0
ans = [2, 1, -1, 3, 2]


In [17]:
def trace_negative_index(nums):
    order, ans, k = [], [], 0
    print("nums =", nums)
    for i, x in enumerate(nums):
        if x == -1:
            k += 1
            if k <= len(order):
                value = order[-k]
                print(f"i={i}: x=-1 | k={k} <= len(order)={len(order)} -> order[-{k}] = {value}")
            else:
                value = -1
                print(f"i={i}: x=-1 | k={k} > len(order)={len(order)} -> -1")
            ans.append(value)
        else:
            order.append(x)
            k = 0
            print(f"i={i}: x={x:>2} | order={order}; k đặt lại = 0")
    print("ans =", ans)
    return ans


assert trace_negative_index([1, 2, -1, -1, -1, 3, -1, -1]) == [2, 1, -1, 3, 2]

nums = [1, 2, -1, -1, -1, 3, -1, -1]
i=0: x= 1 | order=[1]; k đặt lại = 0
i=1: x= 2 | order=[1, 2]; k đặt lại = 0
i=2: x=-1 | k=1 <= len(order)=2 -> order[-1] = 2
i=3: x=-1 | k=2 <= len(order)=2 -> order[-2] = 1
i=4: x=-1 | k=3 > len(order)=2 -> -1
i=5: x= 3 | order=[1, 2, 3]; k đặt lại = 0
i=6: x=-1 | k=1 <= len(order)=3 -> order[-1] = 3
i=7: x=-1 | k=2 <= len(order)=3 -> order[-2] = 2
ans = [2, 1, -1, 3, 2]


Bảng tóm tắt (`nums = [1, 2, -1, -1, -1, 3, -1, -1]`):

| i | x | `order` | `k` | `ans` thêm |
|---|---|---|---|---|
| 0 | 1 | `[1]` | 0 | |
| 1 | 2 | `[1, 2]` | 0 | |
| 2 | -1 | `[1, 2]` | 1 | `order[-1] = 2` |
| 3 | -1 | `[1, 2]` | 2 | `order[-2] = 1` |
| 4 | -1 | `[1, 2]` | 3 | `-1` (3 > 2) |
| 5 | 3 | `[1, 2, 3]` | 0 | |
| 6 | -1 | `[1, 2, 3]` | 1 | `order[-1] = 3` |
| 7 | -1 | `[1, 2, 3]` | 2 | `order[-2] = 2` |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (con trỏ lùi). Đúng class `Solution`, đúng method `lastVisitedIntegers(self, nums: List[int]) -> List[int]`.

In [18]:
from typing import List

class Solution:
    def lastVisitedIntegers(self, nums: List[int]) -> List[int]:
        order: List[int] = []                    # số dương theo thứ tự xuất hiện
        ans: List[int] = []
        ptr = -1                                 # vị trí số "mới nhất chưa dùng" trong đoạn -1 hiện tại
        for x in nums:
            if x == -1:
                ans.append(order[ptr] if ptr >= 0 else -1)
                ptr -= 1
            else:
                order.append(x)
                ptr = len(order) - 1
        return ans


sol = Solution()
for nums, expected in examples:
    print(nums, "->", sol.lastVisitedIntegers(nums))

[1, 2, -1, -1, -1] -> [2, 1, -1]
[1, -1, 2, -1, -1] -> [1, 2, 1]


## 10. Vì sao thuật toán đúng

- **Tương ứng với `seen`:** mô phỏng gốc giữ `seen` với số mới nhất ở đầu. Danh sách `order` (số mới nhất ở cuối) là **đảo ngược** của `seen`. Phần tử thứ `k` của `seen` (`seen[k - 1]`) bằng phần tử `order[-k]` khi `k <= len(order)`.
- **Con trỏ thay cho `k`:** ngay sau một số dương, `ptr = len(order) - 1` trỏ vào số mới nhất (ứng với `k = 1`: `order[-1]`). Mỗi `-1` kế tiếp làm `k` tăng 1, tức phần tử cần lấy lùi sang trái một ô: `ptr -= 1`. Vậy ở `-1` thứ `k` của đoạn, `ptr = len(order) - k`.
- **Điều kiện `-1`:** `ptr >= 0` ⇔ `len(order) - k >= 0` ⇔ `k <= len(order)`, đúng điều kiện đề. Khi vi phạm thì trả `-1`.
- **Đặt lại khi gặp số dương:** `ptr` được gán lại, tương ứng `k = 0` trong đề (chuỗi `-1` liên tiếp bị cắt).
- **Tính chất kiểm được:** số phần tử của `ans` đúng bằng số lượng `-1` của `nums`, và mọi giá trị khác `-1` trong `ans` đều là phần tử của `nums`.

## 11. Phân tích Time / Space Complexity

Gọi `n` là độ dài `nums`.

- **Cách 1:** Time `O(n²)` (do `insert(0, x)`), Space `O(n)`.
- **Cách 2 và Cách 3:** Time **`O(n)`** (mỗi phần tử xử lý `O(1)`), Space **`O(n)`** (`order` và `ans`).
- **Cách 4:** Time `O(n)` đến `O(n²)` tùy cách cắt, Space `O(n)`.
- Mọi cách đều phải đọc từng phần tử nên không thể tốt hơn `O(n)`.

## 12. Bộ test

Ta gom các cách vào từ điển `impls` rồi kiểm tra bằng `assert`.

In [19]:
sol = Solution()
impls = {
    "literal": last_visited_literal,
    "append": last_visited_append,
    "pointer": last_visited_pointer,
    "groupby": last_visited_groupby,
    "submit": sol.lastVisitedIntegers,
}

for nums, expected in examples:
    for key, fn in impls.items():
        assert fn(list(nums)) == expected, (key, nums, fn(list(nums)), expected)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [20]:
def check_all(nums, expected, label):
    for key, fn in impls.items():
        got = fn(list(nums))
        assert got == expected, (label, key, nums, got, expected)
    print("PASS", label, "->", expected)


check_all([-1], [-1], "chỉ có một -1: chưa có số nào")
check_all([5], [], "không có -1 nào -> ans rỗng")
check_all([5, -1], [5], "một số dương rồi một -1")
check_all([-1, -1, -1], [-1, -1, -1], "toàn -1")
check_all([7, -1, -1], [7, -1], "k vượt len(seen) ở -1 thứ hai")
check_all([1, 2, 3, -1, -1, -1], [3, 2, 1], "k đúng bằng len(seen)")
check_all([1, 2, 3, -1, -1, -1, -1], [3, 2, 1, -1], "k vượt len(seen) ở -1 cuối")
check_all([1, -1, 2, -1, 3, -1], [1, 2, 3], "xen kẽ: mỗi -1 là k = 1")
check_all([1, 2, -1, 3, -1, -1, -1], [2, 3, 2, 1], "đoạn -1 thứ hai lùi qua nhiều số")

PASS chỉ có một -1: chưa có số nào -> [-1]
PASS không có -1 nào -> ans rỗng -> []
PASS một số dương rồi một -1 -> [5]
PASS toàn -1 -> [-1, -1, -1]
PASS k vượt len(seen) ở -1 thứ hai -> [7, -1]
PASS k đúng bằng len(seen) -> [3, 2, 1]
PASS k vượt len(seen) ở -1 cuối -> [3, 2, 1, -1]
PASS xen kẽ: mỗi -1 là k = 1 -> [1, 2, 3]
PASS đoạn -1 thứ hai lùi qua nhiều số -> [2, 3, 2, 1]


### 12.2. Tính chất và phản ví dụ

In [21]:
# Tính chất 1: len(ans) == số lượng -1; giá trị khác -1 thuộc nums
random.seed(1)
for _ in range(100):
    nums = [random.choice([-1, -1, random.randint(1, 100)]) for _ in range(random.randint(1, 30))]
    ans = sol.lastVisitedIntegers(nums)
    assert len(ans) == nums.count(-1)
    assert all(v == -1 or v in nums for v in ans)
print("PASS len(ans) = số -1; mọi giá trị là -1 hoặc có trong nums")

# Tính chất 2: nums chỉ có số dương -> ans rỗng
assert sol.lastVisitedIntegers([3, 1, 4, 1, 5]) == []
print("PASS không có -1 -> []")

# Input không bị sửa
nums = [1, -1, 2, -1, -1]
copy = list(nums)
for key, fn in impls.items():
    fn(nums)
assert nums == copy
print("PASS đầu vào không bị sửa")

# Phản ví dụ 1: KHÔNG đặt lại k khi gặp số dương
def sai_khong_reset_k(nums):
    order, ans, k = [], [], 0
    for x in nums:
        if x == -1:
            k += 1
            ans.append(order[-k] if k <= len(order) else -1)
        else:
            order.append(x)                          # thiếu k = 0
    return ans

assert sai_khong_reset_k([1, -1, 2, -1, -1]) != [1, 2, 1]
print("PASS phản ví dụ: quên reset k cho", sai_khong_reset_k([1, -1, 2, -1, -1]), "thay vì [1, 2, 1]")

# Phản ví dụ 2: nhầm seen[k] thay vì seen[k - 1]
def sai_lech_mot(nums):
    seen, ans, k = [], [], 0
    for x in nums:
        if x == -1:
            k += 1
            ans.append(seen[k] if k < len(seen) else -1)
        else:
            seen.insert(0, x)
            k = 0
    return ans

assert sai_lech_mot([1, 2, -1, -1, -1]) != [2, 1, -1]
print("PASS phản ví dụ: lệch một chỉ số cho", sai_lech_mot([1, 2, -1, -1, -1]), "thay vì [2, 1, -1]")

PASS len(ans) = số -1; mọi giá trị là -1 hoặc có trong nums
PASS không có -1 -> []
PASS đầu vào không bị sửa
PASS phản ví dụ: quên reset k cho [1, 1, -1] thay vì [1, 2, 1]
PASS phản ví dụ: lệch một chỉ số cho [1, -1, -1] thay vì [2, 1, -1]


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [22]:
random.seed(2899)
for trial in range(3000):
    n = random.randint(1, 100)
    p_minus = random.choice([0.0, 0.2, 0.5, 0.8, 1.0])
    nums = [-1 if random.random() < p_minus else random.randint(1, 100) for _ in range(n)]
    ref = last_visited_literal(list(nums))
    for key, fn in impls.items():
        assert fn(list(nums)) == ref, (key, nums)
print("PASS 3000 mảng ngẫu nhiên - mọi cách khớp nhau")

PASS 3000 mảng ngẫu nhiên - mọi cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Không đặt lại `k` khi gặp số dương

In [23]:
def sai_quen_reset(nums):
    order, ans, k = [], [], 0
    for x in nums:
        if x == -1:
            k += 1
            ans.append(order[-k] if k <= len(order) else -1)
        else:
            order.append(x)
    return ans


print("sai  :", sai_quen_reset([1, -1, 2, -1, -1]))                  # k cứ tăng dần: [1, 1, -1]
print("đúng :", Solution().lastVisitedIntegers([1, -1, 2, -1, -1]))  # [1, 2, 1]

sai  : [1, 1, -1]
đúng : [1, 2, 1]


**Vì sao sai:** đề đếm **`-1` liên tiếp**. Khi gặp số dương, chuỗi liên tiếp bị cắt nên `k` phải về `0`. Nếu quên, `k` đếm cả những `-1` đã bị ngắt quãng, và đáp án lấy sai phần tử.

### Lỗi 2: Nhầm chỉ số: `seen[k]` thay vì `seen[k - 1]`

In [24]:
def sai_lech_chi_so(nums):
    seen, ans, k = [], [], 0
    for x in nums:
        if x == -1:
            k += 1
            ans.append(seen[k] if k < len(seen) else -1)       # lệch một
        else:
            seen.insert(0, x)
            k = 0
    return ans


print("sai  :", sai_lech_chi_so([1, 2, -1, -1, -1]))
print("đúng :", Solution().lastVisitedIntegers([1, 2, -1, -1, -1]))

sai  : [1, -1, -1]
đúng : [2, 1, -1]


**Vì sao sai:** "phần tử thứ `k`" đếm từ **1**, còn chỉ số Python đếm từ **0** nên là `seen[k - 1]`. Với danh sách đảo chiều (`append`) thì là `order[-k]` (không có `- 1`, vì chỉ số âm đã đếm từ 1).

### Lỗi 3: Truy cập chỉ số trước khi kiểm điều kiện

In [25]:
def sai_khong_kiem_tra(nums):
    seen, ans, k = [], [], 0
    for x in nums:
        if x == -1:
            k += 1
            ans.append(seen[k - 1])                              # quên kiểm tra k <= len(seen)
        else:
            seen.insert(0, x)
            k = 0
    return ans


try:
    sai_khong_kiem_tra([1, -1, -1])
except IndexError as e:
    print("IndexError:", e)
print("đúng :", Solution().lastVisitedIntegers([1, -1, -1]))      # [1, -1]

IndexError: list index out of range
đúng : [1, -1]


**Vì sao sai:** khi `k > len(seen)` đề yêu cầu trả `-1`, không phải truy cập chỉ số. Luôn viết `seen[k - 1] if k <= len(seen) else -1` (kiểm tra trước).

### Lỗi 4: Dùng `list.remove` / `pop(0)` làm thay đổi `seen`

In [26]:
def sai_pop(nums):
    order, ans = [], []
    for x in nums:
        if x == -1:
            ans.append(order.pop() if order else -1)             # pop làm MẤT số đã dùng
        else:
            order.append(x)
    return ans


print("sai  :", sai_pop([1, 2, -1, 3, -1, -1, -1]))    # [2, 3, 1, -1]: số 2 đã bị lấy đi
print("đúng :", Solution().lastVisitedIntegers([1, 2, -1, 3, -1, -1, -1]))   # [2, 3, 2, 1]

sai  : [2, 3, 1, -1]
đúng : [2, 3, 2, 1]


**Vì sao sai:** `pop()` **xóa** số khỏi danh sách; nhưng đề chỉ **đọc** `seen`, không xóa. Sau khi một đoạn `-1` kết thúc, các số cũ vẫn có thể được "thăm" lại ở đoạn `-1` sau (ở ví dụ trên, `2` được dùng ở đoạn đầu rồi vẫn phải xuất hiện lại ở `k = 2` của đoạn sau). Chỉ đọc, không sửa.

### Lỗi 5: Đếm tổng số `-1` thay vì `-1` liên tiếp

In [27]:
def sai_dem_tong(nums):
    order, ans, k = [], [], 0
    for x in nums:
        if x == -1:
            k += 1                                                # không bao giờ reset
            ans.append(order[-1] if order and k == 1 else -1)
        else:
            order.append(x)
    return ans


print("sai  :", sai_dem_tong([1, -1, 2, -1]))                      # [1, -1]: -1 thứ hai bị coi là k = 2
print("đúng :", Solution().lastVisitedIntegers([1, -1, 2, -1]))   # [1, 2]

sai  : [1, -1]
đúng : [1, 2]


**Vì sao sai:** đây là phiên bản khác của Lỗi 1: nếu `k` đếm tổng số `-1` của cả mảng thì các `-1` ở những đoạn khác nhau bị coi là cùng một chuỗi. Mỗi đoạn `-1` liên tiếp phải bắt đầu đếm lại từ `1`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "số gần nhất", "lùi lại `k` bước", "chuỗi liên tiếp bị cắt khi gặp phần tử khác" → nghĩ tới **mảng lưu thứ tự + chỉ số âm** hoặc **con trỏ lùi**.
- Chọn **chiều lưu** sao cho thao tác nhanh: mới nhất ở **cuối** để `append` `O(1)` và `order[-k]`.
- Biến "đếm liên tiếp" luôn cần **quy tắc đặt lại** rõ ràng.
- Kiểm tra `k <= len(order)` **trước** khi truy cập chỉ số.
- Bài mô phỏng: làm đúng theo đề trước (Cách 1), rồi mới tối ưu, và đối chiếu hai bản bằng ngẫu nhiên.
- `ptr` giảm dần: khi `ptr < 0` mọi `-1` còn lại của đoạn đều trả `-1`.

## 15. Biến thể của bài

1. Thay `-1` bằng **ký tự `"prev"`** trong danh sách chuỗi (bài "Last Visited Integers" có phiên bản `words` trong một đợt thi khác).
2. Lấy phần tử thứ `k` từ **đầu** thay vì từ gần nhất: tức đọc `order[k - 1]`.
3. Mỗi `-1` chỉ lùi **một** bước (không cộng dồn): luôn trả số dương gần nhất.
4. Trả về **chỉ số** của số được tìm thấy thay vì giá trị.
5. Mảng có thêm giá trị `0` mang nghĩa "xóa số mới nhất".
6. Xử lý luồng dữ liệu (stream) trực tuyến: mỗi lần nhận một phần tử, trả ngay đáp án nếu là `-1`.

Ta thử biến thể 3 và 4:

In [28]:
def last_visited_only_recent(nums):
    """Biến thể 3: mỗi -1 luôn trả số dương gần nhất (không cộng dồn k)."""
    last, ans = -1, []
    for x in nums:
        if x == -1:
            ans.append(last)
        else:
            last = x
    return ans


def last_visited_indices(nums):
    """Biến thể 4: trả về chỉ số (trong nums) của số được thăm; -1 nếu không có."""
    positions, ans, ptr = [], [], -1
    for i, x in enumerate(nums):
        if x == -1:
            ans.append(positions[ptr] if ptr >= 0 else -1)
            ptr -= 1
        else:
            positions.append(i)
            ptr = len(positions) - 1
    return ans


print(last_visited_only_recent([1, 2, -1, -1, -1]))      # [2, 2, 2]
print(last_visited_indices([1, 2, -1, -1, -1]))          # [1, 0, -1]

[2, 2, 2]
[1, 0, -1]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng `insert(0, x)` | làm đúng từng chữ của đề | `O(n²)` | `O(n)` | hiểu đề, đối chiếu |
| 2. `append` + `order[-k]` | đảo chiều danh sách, đếm `k` | `O(n)` | `O(n)` | dễ hiểu, nộp bài |
| 3. Con trỏ lùi `ptr` | thay đếm `k` bằng lùi con trỏ | `O(n)` | `O(n)` | **cách nộp bài**, gọn nhất |
| 4. `groupby` từng đoạn `-1` | `order[::-1][:L]` đệm `-1` | `O(n)` đến `O(n²)` | `O(n)` | khi muốn nghĩ theo "đoạn" |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `[10, 20, 30][-1]`, `[10, 20, 30][-3]`, `[1, 2, 3][::-1]`, `Solution().lastVisitedIntegers([4, -1, -1])`, `Solution().lastVisitedIntegers([-1, 4])`.

**Bài 2 (tính tay).** Với `nums = [3, 5, -1, -1, 7, -1, -1, -1]`, lập bảng `order`, `k` (hoặc `ptr`), giá trị thêm vào `ans` ở từng bước rồi viết `ans`.

**Bài 3 (điền TODO).** Hoàn thành `last_visited_todo(nums)` theo Cách 3 (con trỏ lùi).

In [29]:
def last_visited_todo(nums):
    order = []
    ans = []
    ptr = -1
    for x in nums:
        if x == -1:
            # TODO 1: nếu ptr >= 0 thì thêm order[ptr] vào ans, ngược lại thêm -1
            # TODO 2: lùi con trỏ: ptr -= 1
            pass
        else:
            # TODO 3: thêm x vào order
            # TODO 4: đặt ptr = len(order) - 1
            pass
    return ans

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `last_visited_deque(nums)` dùng `deque.appendleft` để giữ `seen` đúng như đề (mới nhất đứng đầu), mà không nhìn code mẫu. Kiểm tra với hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `last_visited_words(words)` cho mảng chuỗi, trong đó phần tử `"prev"` đóng vai trò `-1` (các chuỗi khác là "số dương"); trả `"-1"` (chuỗi) khi không đủ. (b) Viết `max_k_reached(nums)` trả về **giá trị `k` lớn nhất** mà đoạn `-1` liên tiếp nào đó đạt tới (độ dài đoạn `-1` dài nhất).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `30`, `10`, `[3, 2, 1]`, `[4, -1]`, `[-1]`. Bài 2: mỗi lần gặp số dương thì `k` về 0 (hoặc `ptr` đặt lại).
- **Mức 2 (vừa):** Bài 2: `order = [3, 5]`; `-1` đầu: `5`; `-1` thứ hai: `3`; rồi `7` (đặt lại, `order = [3, 5, 7]`); ba `-1` tiếp theo: `7`, `5`, `3`. Vậy `ans = [5, 3, 7, 5, 3]`. Bài 3: bốn TODO đã ghi trong comment. Bài 4: `seen = deque()`, `seen.appendleft(x)`, `seen[k - 1] if k <= len(seen) else -1`.
- **Mức 3 (sâu):** Bài 5(a): thay điều kiện `x == -1` bằng `w == "prev"` và `-1` bằng `"-1"`. Bài 5(b): duyệt đếm run: `cur = cur + 1 if x == -1 else 0`, cập nhật `best = max(best, cur)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [30]:
# Bài 1
assert [10, 20, 30][-1] == 30 and [10, 20, 30][-3] == 10
assert [1, 2, 3][::-1] == [3, 2, 1]
assert Solution().lastVisitedIntegers([4, -1, -1]) == [4, -1]
assert Solution().lastVisitedIntegers([-1, 4]) == [-1]

# Bài 2
assert Solution().lastVisitedIntegers([3, 5, -1, -1, 7, -1, -1, -1]) == [5, 3, 7, 5, 3]

# Bài 3
def last_visited_todo_ans(nums):
    order = []
    ans = []
    ptr = -1
    for x in nums:
        if x == -1:
            ans.append(order[ptr] if ptr >= 0 else -1)
            ptr -= 1
        else:
            order.append(x)
            ptr = len(order) - 1
    return ans

for nums, expected in examples:
    assert last_visited_todo_ans(nums) == expected

# Bài 4
def last_visited_deque(nums):
    seen = deque()
    ans = []
    k = 0
    for x in nums:
        if x == -1:
            k += 1
            ans.append(seen[k - 1] if k <= len(seen) else -1)
        else:
            seen.appendleft(x)
            k = 0
    return ans

for nums, expected in examples:
    assert last_visited_deque(nums) == expected

# Bài 5(a)
def last_visited_words(words):
    order, ans, ptr = [], [], -1
    for w in words:
        if w == "prev":
            ans.append(order[ptr] if ptr >= 0 else "-1")
            ptr -= 1
        else:
            order.append(w)
            ptr = len(order) - 1
    return ans

assert last_visited_words(["a", "b", "prev", "prev", "prev"]) == ["b", "a", "-1"]

# Bài 5(b)
def max_k_reached(nums):
    best = cur = 0
    for x in nums:
        cur = cur + 1 if x == -1 else 0
        best = max(best, cur)
    return best

assert max_k_reached([3, 5, -1, -1, 7, -1, -1, -1]) == 3
assert max_k_reached([1, 2, 3]) == 0
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (mảng lưu thứ tự + con trỏ lùi)

```python
from typing import List

class Solution:
    def lastVisitedIntegers(self, nums: List[int]) -> List[int]:
        order, ans, ptr = [], [], -1
        for x in nums:
            if x == -1:
                ans.append(order[ptr] if ptr >= 0 else -1)
                ptr -= 1
            else:
                order.append(x)
                ptr = len(order) - 1
        return ans
```

**Checklist**

- [ ] Đặt lại `k` (hoặc gán lại `ptr`) mỗi khi gặp **số dương**?
- [ ] `k` đếm `-1` **liên tiếp** (không phải tổng số `-1` của cả mảng)?
- [ ] Đúng chỉ số: `seen[k - 1]` (danh sách mới nhất đứng đầu) hoặc `order[-k]` (mới nhất đứng cuối)?
- [ ] Kiểm tra `k <= len(...)` / `ptr >= 0` **trước** khi truy cập, trả `-1` nếu vượt?
- [ ] Không xóa phần tử khỏi danh sách (`pop`, `remove`) trong lúc đọc?
- [ ] Đã thử: chỉ toàn `-1`, không có `-1`, `k` đúng bằng `len`, `k` vượt `len`?
- [ ] Skeleton đúng `class Solution` / `def lastVisitedIntegers(self, nums: List[int]) -> List[int]`?